# NOTEBOOK 12 — JOB MARKET ALIGNMENT

## Project
Job Market Analyzer for Career Readiness in South Africa

## Purpose
Compare candidate skills with the South African job market to calculate Job-Market Alignment.

## Position in Pipeline

Previous: Notebook 11 — Career Readiness

Current: Notebook 12

Next: Notebook 13 — Dashboard & Final Integration

## Main Objective
Calculate how closely the candidate's profile matches current market requirements for the selected career.

## Input
- `data/processed/sa_it_jobs_matched.csv` — matched job dataset
- `data/processed/candidate_profile.json` — candidate profile

## Output
- Job-Market Alignment Score
- Market skill gaps
- Location and work-type analysis

---

# NOTEBOOK ROADMAP

STEP 1 — Define Market Alignment
STEP 2 — Identify Relevant Jobs
STEP 3 — Define Relevant Market Segment
STEP 4 — Extract Market Skills
STEP 5 — Calculate Skill Demand
STEP 6 — Compare Candidate Skills
STEP 7 — Identify Market Skill Gaps
STEP 8 — Analyze Target-Role Demand
STEP 9 — Analyze Location and Work-Type Patterns
STEP 10 — Calculate Market Alignment
STEP 11 — Interpret Results
STEP 12 — Identify Candidate Priorities
STEP 13 — Save Market Analysis

---

# STEP 1 — Define Market Alignment

## Purpose
Establish what job-market alignment means.

## Definition
Job-Market Alignment answers: How closely does this candidate's profile match what employers are currently requesting for the selected career?

## Method
Compare candidate skills with market demand using weighted coverage.

---

# STEP 2 — Identify Relevant Jobs

## Purpose
Filter jobs relevant to the candidate's target role.

## Method
Filter by target role keywords.

In [ ]:
# Load data
import json

with open('data/processed/candidate_profile.json', 'r') as f:
    candidate = json.load(f)

df = pd.read_csv('data/processed/sa_it_jobs_matched.csv')

# Filter relevant jobs
target_role = "data analyst"
relevant_jobs = df[df['Job_Title_Normalized'].str.contains(target_role, case=False, na=False)]

print(f"Relevant jobs for '{target_role}': {len(relevant_jobs)}")
print(f"Total jobs in dataset: {len(df)}")

---

# STEP 3 — Define Relevant Market Segment

## Purpose
Define the market segment for analysis.

## Method
Use the relevant jobs as the market segment.

In [ ]:
# Market segment
market_segment = relevant_jobs

print("Market Segment:")
print("="*60)
print(f"  Jobs in segment: {len(market_segment)}")
print(f"  Unique companies: {market_segment['Company'].nunique()}")
print(f"  Provinces: {market_segment['Province'].nunique()}")

---

# STEP 4 — Extract Market Skills

## Purpose
Extract all skills demanded in the market segment.

## Method
Extract and count individual skills.

In [ ]:
# Extract market skills
all_market_skills = []
for skills in market_segment['Skills'].dropna():
    all_market_skills.extend([s.strip().lower() for s in str(skills).split(',')])

market_skill_counts = pd.Series(all_market_skills).value_counts()
total_mentions = len(all_market_skills)

print("Market Skills:")
print("="*60)
print(f"  Total skill mentions: {total_mentions}")
print(f"  Unique skills: {len(market_skill_counts)}")
print(f"\nTop 10 market skills:")
print(market_skill_counts.head(10).to_string())

---

# STEP 5 — Calculate Skill Demand

## Purpose
Calculate the demand percentage for each skill.

## Formula
Skill Demand % = (Number of jobs requiring skill / Total jobs in segment) × 100

## Method
Calculate demand for each skill.

In [ ]:
# Calculate skill demand
total_jobs = len(market_segment)
skill_demand = {}

for skill in market_skill_counts.index:
    count = market_segment['Skills'].str.contains(skill, case=False, na=False).sum()
    skill_demand[skill] = (count / total_jobs) * 100

skill_demand_df = pd.DataFrame({
    'Skill': list(skill_demand.keys()),
    'Demand %': list(skill_demand.values())
}).sort_values('Demand %', ascending=False)

print("Skill Demand:")
print("="*60)
print(skill_demand_df.head(15).to_string(index=False))

---

# STEP 6 — Compare Candidate Skills

## Purpose
Compare candidate skills with market demand.

## Method
Identify which demanded skills the candidate has.

In [ ]:
# Compare candidate skills
candidate_skills = set(s.lower() for s in candidate['skills'])
demanded_skills = set(skill_demand.keys())

matched_skills = demanded_skills & candidate_skills
missing_skills = demanded_skills - candidate_skills

print("Candidate vs Market Skills:")
print("="*60)
print(f"  Demanded skills: {len(demanded_skills)}")
print(f"  Candidate skills: {len(candidate_skills)}")
print(f"  Matched: {len(matched_skills)}")
print(f"  Missing: {len(missing_skills)}")

---

# STEP 7 — Identify Market Skill Gaps

## Purpose
Identify skills the candidate lacks that are in high demand.

## MethodFind missing skills with highest demand.

In [ ]:
# Identify skill gaps
skill_gaps = []
for skill in missing_skills:
    demand = skill_demand.get(skill, 0)
    skill_gaps.append({'Skill': skill, 'Demand %': demand})

skill_gaps_df = pd.DataFrame(skill_gaps).sort_values('Demand %', ascending=False)

print("Market Skill Gaps:")
print("="*60)
print(skill_gaps_df.head(10).to_string(index=False))

# Visualization
plt.figure(figsize=(12, 6))
top_gaps = skill_gaps_df.head(10)
sns.barplot(x='Demand %', y='Skill', data=top_gaps, palette='Reds')
plt.title('Top 10 Market Skill Gaps', fontsize=14, fontweight='bold')
plt.xlabel('Market Demand %')
plt.ylabel('Skill')
plt.tight_layout()
plt.savefig('outputs/market_skill_gaps.png', dpi=300, bbox_inches='tight')
plt.show()

---

# STEP 8 — Analyze Target-Role Demand

## PurposeAnalyze demand for the target role.

## MethodCalculate demand metrics for the target role.

In [ ]:
# Target role demand
target_role_demand = len(market_segment)
total_market = len(df)
role_percentage = (target_role_demand / total_market) * 100

print("Target Role Demand:")
print("="*60)
print(f"  Target role jobs: {target_role_demand}")
print(f"  Total market: {total_market}")
print(f"  Role percentage: {role_percentage:.1f}%")

---

# STEP 9 — Analyze Location and Work-Type Patterns

## PurposeAnalyze geographic and work-type patterns in the market.

## MethodCalculate distributions for the market segment.

In [ ]:
# Location and work-type analysis
print("Location and Work-Type Patterns:")
print("="*60)

print("\nProvince Distribution:")
print(market_segment['Province'].value_counts().to_string())

print("\nWork Type Distribution:")
print(market_segment['Remote'].value_counts().to_string())

print("\nContract Type Distribution:")
print(market_segment['Contract_Type'].value_counts().to_string())

---

# STEP 10 — Calculate Market Alignment

## Purpose
Calculate the overall market alignment score.

## Formula
Market Skill Coverage = (Sum of demand weights for matched skills / Sum of demand weights for all skills) × 100

## Method
Calculate weighted coverage.

In [ ]:
# Calculate market alignment
total_demand_weight = sum(skill_demand.values())
matched_demand_weight = sum(skill_demand.get(skill, 0) for skill in matched_skills)

market_alignment = (matched_demand_weight / total_demand_weight * 100) if total_demand_weight > 0 else 0

print("Market Alignment:")
print("="*60)
print(f"  Total demand weight: {total_demand_weight:.1f}")
print(f"  Matched demand weight: {matched_demand_weight:.1f}")
print(f"  Market Alignment: {market_alignment:.1f}%")

---

# STEP 11 — Interpret Results

## Result Interpretation

### What happened?
Candidate skills were compared with market demand to calculate alignment.

### What does it mean?
The market alignment score indicates how well the candidate's skills match current market requirements.

### Why does it matter?
Market alignment helps identify skill gaps and career development priorities.

### What decision does it support?
The analysis guides personalized skill development recommendations.

### What limitation should be considered?
The analysis is based on a snapshot of the job market and may not capture all available positions.

---

# STEP 12 — Identify Candidate Priorities

## Purpose
Identify top priorities for the candidate.

## MethodRank missing skills by demand.

In [ ]:
# Candidate priorities
priorities = skill_gaps_df.head(5)['Skill'].tolist()

print("Candidate Priorities:")
print("="*60)
for i, skill in enumerate(priorities, 1):
    demand = skill_demand.get(skill, 0)
    print(f"  {i}. {skill} (Demand: {demand:.1f}%)")

---

# STEP 13 — Save Market Analysis

## Purpose
Save the market analysis results.

## Method
Save to JSON for use in the dashboard.

In [ ]:
# Save market analysis
market_analysis = {
    'market_alignment_score': round(market_alignment, 2),
    'target_role': target_role,
    'relevant_jobs': len(market_segment),
    'total_market': len(df),
    'role_percentage': round(role_percentage, 2),
    'matched_skills': list(matched_skills),
    'missing_skills': list(missing_skills),
    'skill_gaps': skill_gaps_df.head(10).to_dict('records'),
    'priorities': priorities,
    'province_distribution': market_segment['Province'].value_counts().to_dict(),
    'work_type_distribution': market_segment['Remote'].value_counts().to_dict(),
    'contract_type_distribution': market_segment['Contract_Type'].value_counts().to_dict()
}

with open('outputs/market_alignment.json', 'w') as f:
    json.dump(market_analysis, f, indent=2)

print("Market analysis saved to: outputs/market_alignment.json")

---

# FINAL NOTEBOOK RESULTS

## What We Did
Compared candidate skills with the South African job market to calculate Job-Market Alignment.

## Key Results
1. Identified relevant jobs for the target role
2. Calculated skill demand percentages
3. Identified market skill gaps
4. Analyzed location and work-type patterns
5. Calculated market alignment score

## Result Interpretation
The market alignment score indicates how well the candidate's skills match current market requirements.

## Data Science Decisions
- Used weighted coverage for market alignment
- Analyzed only relevant market segment
- Identified priorities based on demand

## Outputs Produced
- Market analysis: outputs/market_alignment.json
- Visualization: outputs/market_skill_gaps.png

## Limitations
- Based on snapshot of job market
- May not capture all available positions

## Connection to the Next Notebook
Notebook 13 will integrate all components into a Streamlit dashboard for the final user experience.